In [2]:
import sys
from pathlib import Path

# This file is inside notebooks/, so project root is one level up
PROJECT_ROOT = Path.cwd().parent

# Add root to Python path so "import core" works
sys.path.insert(0, str(PROJECT_ROOT))

print("Notebook cwd:", Path.cwd())
print("Project root added:", PROJECT_ROOT)


Notebook cwd: c:\Users\fatem\OneDrive\Desktop\capstone\Agentic-Crypto-Return-Service\notebooks
Project root added: c:\Users\fatem\OneDrive\Desktop\capstone\Agentic-Crypto-Return-Service


In [3]:
import sklearn
import numpy as np
import pandas as pd

from core.models.probabilistic_quantile import (
    load_features_csv,
    add_next_day_target,
    time_split,
    fit_quantile_models,
    predict_quantiles,
    sample_from_quantiles,
    var_cvar
)

PATH = "../data/processed/features/BTC_features.csv"

FEATURES = [
    "log_ret_1d", "log_ret_5d", "log_ret_10d",
    "vol_7d", "vol_30d",
    "risk_adj_ret_1d", "vol_ratio_7d_30d",
    "drawdown_30d"
]

df = load_features_csv(PATH)
df = add_next_day_target(df, ret_col="log_ret_1d")
train, test = time_split(df, train_frac=0.8)

bundle = fit_quantile_models(
    train_df=train,
    feature_cols=FEATURES,
    target_col="target_log_ret_1d",
    quantiles=[0.05, 0.50, 0.95],
)

# Pick the most recent row in the dataset (last available "today")
row = df.tail(1).copy()

qpred = predict_quantiles(bundle, row)
print("Predicted quantiles for next-day return:")
print(qpred)

samples = sample_from_quantiles(qpred, quantiles=bundle.quantiles, n_samples=5000, seed=42)
VaR_5, CVaR_5 = var_cvar(samples, alpha=0.05)

print("\n1-day VaR(5%):", VaR_5)
print("1-day CVaR(5%):", CVaR_5)


Predicted quantiles for next-day return:
       q_0.05    q_0.50    q_0.95
2199 -0.05552  0.001264  0.061498

1-day VaR(5%): -0.055399158669447245
1-day CVaR(5%): -0.055518136067763936


retrain with more quantiles

In [4]:
# More quantiles = better approximation of the whole distribution (especially tails)
quantiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

bundle = fit_quantile_models(
    train_df=train,
    feature_cols=FEATURES,
    target_col="target_log_ret_1d",
    quantiles=quantiles,
    random_state=42,
    n_estimators=120,     # keep moderate to avoid long runtimes
    learning_rate=0.05,
    max_depth=2,
)

today = df.tail(1).copy()
qpred = predict_quantiles(bundle, today)
print(qpred)


        q_0.01    q_0.05    q_0.10    q_0.25   q_0.50   q_0.75    q_0.90  \
2199 -0.148164 -0.072351 -0.046865 -0.014574  0.00154  0.01692  0.044624   

       q_0.95    q_0.99  
2199  0.06176  0.096232  


Why this is strong

Asymmetry: downside tail is much fatter than upside → realistic for crypto

Median near zero: matches efficient-market intuition

Wide tails: reflect volatility clustering

This is a regime-conditioned distribution, not a historical average.

## Compute 1-day VaR/CVaR for today again:

 typically CVaR is more negative than VaR (meaningfully)

In [5]:
samples = sample_from_quantiles(qpred, quantiles=bundle.quantiles, n_samples=10000, seed=42)
VaR_5, CVaR_5 = var_cvar(samples, alpha=0.05)

print("1-day VaR (5%):", VaR_5)
print("1-day CVaR (5%):", CVaR_5)


1-day VaR (5%): -0.07410864261512204
1-day CVaR (5%): -0.11742229662546787


What it means in plain language

5% of the time, tomorrow is worse than −7.4%

When it is in that worst 5%, the average loss is closer to −11.7%

## VaR/CVaR time series on the TEST set

This computes risk metrics for each day in the test window (based on that day’s predicted distribution). (CVaR_5 <= VaR_5) should be very close to 1.0

CVaR should be noticeably more negative than VaR on average

In [6]:
results = []

for i in range(len(test)):
    row = test.iloc[[i]]  # keep as DataFrame (single row)
    qpred_i = predict_quantiles(bundle, row)
    samples_i = sample_from_quantiles(qpred_i, quantiles=bundle.quantiles, n_samples=3000, seed=123)

    var5, cvar5 = var_cvar(samples_i, alpha=0.05)

    results.append({
        "date": row["date"].iloc[0],
        "VaR_5": var5,
        "CVaR_5": cvar5,
        "q_0.50": float(qpred_i["q_0.50"].iloc[0]),
        "realized_next_day_ret": float(row["target_log_ret_1d"].iloc[0])
    })

risk_df = pd.DataFrame(results).sort_values("date").reset_index(drop=True)
risk_df.head(), risk_df.tail()


(        date     VaR_5    CVaR_5    q_0.50  realized_next_day_ret
 0 2024-11-25 -0.056106 -0.084700  0.001207              -0.012070
 1 2024-11-26 -0.043758 -0.064812  0.001441               0.042329
 2 2024-11-27 -0.043000 -0.058107 -0.000431              -0.003236
 3 2024-11-28 -0.043000 -0.058107  0.000994               0.018736
 4 2024-11-29 -0.042992 -0.058106 -0.000466              -0.010443,
           date     VaR_5    CVaR_5    q_0.50  realized_next_day_ret
 435 2026-02-03 -0.064352 -0.100657  0.011603              -0.035171
 436 2026-02-04 -0.068471 -0.122710  0.011814              -0.152334
 437 2026-02-05 -0.084992 -0.095585  0.003112               0.118003
 438 2026-02-06 -0.058663 -0.065229 -0.008558              -0.018213
 439 2026-02-07 -0.071319 -0.117319  0.001540               0.023808)

In [7]:
print("CVaR should be <= VaR most of the time:")
print((risk_df["CVaR_5"] <= risk_df["VaR_5"]).mean())

print("\nSummary:")
print(risk_df[["VaR_5","CVaR_5","q_0.50"]].describe())


CVaR should be <= VaR most of the time:
1.0

Summary:
            VaR_5      CVaR_5      q_0.50
count  440.000000  440.000000  440.000000
mean    -0.040217   -0.063495    0.000590
std      0.011170    0.013628    0.002379
min     -0.084992   -0.136890   -0.008558
25%     -0.045568   -0.067531   -0.000961
50%     -0.040963   -0.058479   -0.000026
75%     -0.028168   -0.053134    0.001226
max     -0.026149   -0.051262    0.011814


In [10]:
from core.models.probabilistic_quantile import fit_quantile_models, compute_var_cvar_timeseries
import inspect

print("Default quantiles:", inspect.signature(fit_quantile_models))
print("Has compute_var_cvar_timeseries:", callable(compute_var_cvar_timeseries))


ImportError: cannot import name 'compute_var_cvar_timeseries' from 'core.models.probabilistic_quantile' (c:\Users\fatem\OneDrive\Desktop\capstone\Agentic-Crypto-Return-Service\core\models\probabilistic_quantile.py)